# 15 — Auditoria automática dos dados de treino

O notebook 14 terminou sem um limiar global elegível. Antes de outro treino,
vamos investigar caixas, cobertura, repetições, contexto visual e coerência
semântica dos rótulos. **Sem preencher fichas e sem alterar os originais.**

| Fonte | Escopo desta auditoria |
|---|---|
| Home TRAIN | Todas as 1.824 imagens do pacote atual |
| Open Images TRAIN | Todas as 5.000 imagens dos cinco lotes usados no piloto |
| Histórico Open Images | Distribuição documental dos 80.000 IDs de treino |

Todas as **12 classes** e as linhas de rótulo das imagens elegíveis são
auditadas. Bloqueios e quarentenas ficam registrados. Mesas, cadeiras, plantas e `wardrobe` recebem destaque nos relatórios.
O detector B e o CLIP congelado fornecem **sinais por IA**, sem confirmar que
um rótulo está certo ou errado. O recorte de cinco lotes não é representativo
dos 80 mil exemplos e não constitui um dataset novo aprovado.

As validações, Home TEST, Open Images TEST e as 300 reservadas não entram
na inferência. Seus metadados de identidade já preservados servem apenas
para bloquear coincidências com o treino. Nenhum ZIP de teste/reserva é baixado.


## 1. Preparar e executar no Kaggle

Importe este notebook, ative **Internet** e prefira **GPU T4**. Habilite o
Secret existente **GDRIVE_RCLONE_CONFIG_B64**. Use **Save Version → Save & Run All**.
Os caminhos estão preenchidos; não é preciso anexar pacotes antigos ou mudar células.

CPU também é aceita, mas a análise dos recortes tende a demorar mais. O código
mostra contagens, velocidade e estimativa de tempo após começar a inferência.
Os caches são salvos por blocos no Drive e recuperados automaticamente quando
as entradas, o código, os pesos e o ambiente coincidem.

Ao término, os resultados estão em **pacote_auditoria_15.zip**. Não há formulário para preencher.


In [ ]:
%pip install -q ultralytics==8.4.150 pillow scipy numpy ftfy regex tqdm packaging setuptools wheel
%pip install -q --no-deps --no-build-isolation git+https://github.com/openai/CLIP.git@d05afc436d78f1c48dc0dbf8e5980a9d471f35f6


In [ ]:
import os
os.environ['YOLO_OFFLINE']='true'
import sys, json, hashlib, shutil, platform, random, zipfile, traceback
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import numpy as np
import torch, ultralytics
from ultralytics import settings
from IPython.display import display, FileLink

SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
DEVICE=0 if torch.cuda.is_available() else 'cpu'
if ultralytics.__version__!='8.4.150':raise RuntimeError('Reinicie a sessão após instalar ultralytics==8.4.150.')
settings.update({k:False for k in ['sync','hub','platform','wandb','comet','mlflow','neptune','clearml','dvc','tensorboard'] if k in settings})
if 'NOME_AUDITORIA15' not in globals():
    NOME_AUDITORIA15='auditoria15_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
PASTA_LOCAL=Path('/tmp/tcc_auditoria_15')/NOME_AUDITORIA15
PASTA_RESULTADOS=Path('/kaggle/working/resultados_tcc')/NOME_AUDITORIA15
PASTA_LOCAL.mkdir(parents=True,exist_ok=True);PASTA_RESULTADOS.mkdir(parents=True,exist_ok=True)
ENVIAR_RESULTADOS_AO_DRIVE=True
print('Dispositivo:',torch.cuda.get_device_name(0) if DEVICE==0 else 'CPU')
print('Execução:',NOME_AUDITORIA15)


## 2. Carregar o código e conferir o analisador semântico

CLIP ViT-B/32 é usado congelado, sem ajuste nas nossas imagens. As descrições
incluem as 12 classes e alternativas como estante, rack, cômoda e armário
genérico. Cada caixa recebe dois recortes com padding: a caixa e um contexto
20% maior. A comparação usa similaridade de cosseno, não probabilidade de acerto.

Uma divergência forte exige o mesmo conceito alternativo nos dois recortes,
similaridade de pelo menos 0,20 e margem de pelo menos 0,03 sobre o rótulo.
São heurísticas fixadas antes da execução, sem calibração de taxa de erro.
Recortes com lado menor que 8 pixels são registrados como insuficientes.


In [ ]:
EXPECTED = {'baseline': '4170653e0c0a604f85b6747e42b3fd2ff5ddb765ee4762915dfc9f3d33d322d9', 'candidato': '97040f91876d8835a67a7db287d3e1c5f129a4068e36dd4483d947f62b260c5f', 'politica': '1aa6b13cb68d8f30a2f388c3577a61a12dcac4c604581877091d3fbdc6042845', 'home_zip': 'ada7fbee37b83f673b1b76bc57643d1fe4648941e0e8a7b895282216aed47676', 'test_home': 'f48be0a2b3a93e59259b1ae0d3655028951bf270ef7f4e7916a9424530951a89', 'preparacao': '9865042654c51e83955c2aeb186a1b5e27e1028a5c56f1f163e2891abc0bec49', 'divisao_10': 'a5dd35ff484dda77e8988aefdc961f05d07ea409d1336cbe90c4a5420632bab9', 'manifesto_pacotes': '655094dea28755a7bddc817ae66660d06fab28ccb8f9d140d3ffebb07f03a0d4', 'pacote_11': 'a3f3fb500cef83fc4890f6bbec6725bbf5e080503fd0478af3dd84d6f5ef6e93', 'dataset_novo': '695b445e15c905642e2b276801b71132ced28e5eaee92d9fbcdd608f582f15a0', 'zip_novo': '5af596546e2ad734c7b663f4567de2087d7fb42b41527d4852affdb4e7d2b96c', 'codigo_11': {'diag11_core.py': '29e7c52fced44ab4898edfbc46477176587f290d2fbdacad4de5f595d4187670', 'diag11_data.py': 'c9f137d4dcc8c8f5fb156aaf25cddf6707d24e74567c05237332c2144e6e2f98', 'diag11_reports.py': '0068af8fd70b616cb642e9fb8a75d503cdc003f18fe79d64fd487bd7822d85e2'}, 'pacote_12': '1c3a6016099c2bb23aca29396286497c7973f00136c1afbb0bbc62e45aa20650', 'codigo_12': {'diag11_core.py': '29e7c52fced44ab4898edfbc46477176587f290d2fbdacad4de5f595d4187670', 'diag11_data.py': 'c9f137d4dcc8c8f5fb156aaf25cddf6707d24e74567c05237332c2144e6e2f98', 'diag11_reports.py': '0068af8fd70b616cb642e9fb8a75d503cdc003f18fe79d64fd487bd7822d85e2', 'exp10_partial.py': '86a36f93aa643853f1268dcc133bed7e7344dea269b9d1d58cf54095538fffae', 'exp12_protocol.py': '257d09b0b2aafe94da28bdcb59859778ffb31e25601db75d1058092b943b5759', 'exp12_data.py': '0a9de7e99d302a2690f18f51924f4dc80b32861c9d2149ccfe0572ec75499f30', 'exp12_training.py': '6c51c9965336b2f00bd9b282e2159c0b63f07c5c16859da78da2818bd4202fdc', 'exp12_reports.py': '120609247939189f895c01cdfd1411e4d2b416d1e293cb26851d63d04835b49a'}, 'B_e03': 'b683e1ef5fac4f8ddb4fad2f610e10dcb33bc658ea72593b8088652dbdd3c21c', 'regra_12_sha256': '04c6287c2c4adb8c43baf28c10acbaa1a599198b905d48808a8f5f67f39a4909'}
CLIP_ARQUIVOS_SHA256 = {'__init__.py': 'd7930eb290ebed432116795c91473200d098451a94c0aedd4ae0f934f97955e8', 'bpe_simple_vocab_16e6.txt.gz': '924691ac288e54409236115652ad4aa250f48203de50a9e4722a6ecd48d6804a', 'clip.py': '9540f200fbf8145479fa655382a56dab048d238cc698b9cbd8df3b6d86d3f1b6', 'model.py': '9902cbe5ee90a1da2aa3e6f043e8a23dc1f8831193b963785c9af03d5c7bef2c', 'simple_tokenizer.py': 'd1b09f10ed3d1e2c343619cb98cbfacb92363dd8607526a43aa7fe2d60d15bad'}
MODULOS_EMBUTIDOS = {'audit15_core.py': '"""Auditoria de TRAIN: identidade, caixas, cobertura e duplicações. Sem treinamento."""\nfrom __future__ import annotations\n\nimport csv\nimport gzip\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport stat\nimport time\nimport zipfile\nfrom collections import Counter, defaultdict\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nfrom PIL import Image\nfrom scipy.fft import dctn\n\nVERSION = \'15-auditoria-train-1.0.0\'\nNAMES = [\'bed\', \'sofa\', \'chair\', \'table\', \'lamp\', \'tv\', \'laptop\',\n         \'wardrobe\', \'window\', \'door\', \'potted plant\', \'photo frame\']\nALL_KNOWN = (1 << len(NAMES)) - 1\nFOCUS = {2, 3, 7, 10}\n\n\ndef canonical(x):\n    return json.dumps(x, sort_keys=True, ensure_ascii=False, separators=(\',\', \':\'), allow_nan=False)\n\n\ndef digest(x):\n    return hashlib.sha256(canonical(x).encode()).hexdigest()\n\n\ndef sha_file(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for part in iter(lambda: f.read(2**20), b\'\'):\n            h.update(part)\n    return h.hexdigest()\n\n\ndef save_json(path, value):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + \'.tmp\')\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding=\'utf-8\')\n    tmp.replace(path)\n\n\ndef save_csv(path, rows, fields=None):\n    rows = list(rows); path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    fields = fields or list(dict.fromkeys(k for r in rows for k in r)) or [\'sem_registros\']\n    with path.open(\'w\', encoding=\'utf-8-sig\', newline=\'\') as f:\n        w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(rows)\n\n\ndef save_gz(path, obj):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + \'.tmp\')\n    with tmp.open(\'wb\') as f, gzip.GzipFile(fileobj=f, mode=\'wb\', mtime=0) as g:\n        g.write(canonical(obj).encode())\n    tmp.replace(path)\n\n\ndef load_gz(path):\n    with gzip.open(path, \'rt\', encoding=\'utf-8\') as f:\n        return json.load(f)\n\n\ndef safe_member(name):\n    p = PurePosixPath(name)\n    if (not name or p.is_absolute() or \'..\' in p.parts or \'\\\\\' in name\n            or \':\' in p.parts[0] or \'\\x00\' in name):\n        raise ValueError(\'Caminho inseguro no ZIP: \' + str(name))\n    return p\n\n\ndef inspect_zip(z):\n    names = z.namelist()\n    if len(names) != len(set(names)):\n        raise ValueError(\'ZIP com nomes duplicados.\')\n    for info in z.infolist():\n        safe_member(info.filename)\n        if stat.S_ISLNK(info.external_attr >> 16):\n            raise ValueError(\'Link simbólico não permitido no ZIP.\')\n    return set(names)\n\n\ndef checked_read(z, member, expected):\n    safe_member(member)\n    raw = z.read(member)\n    if hashlib.sha256(raw).hexdigest() != expected:\n        raise ValueError(\'Conteúdo diferente do manifesto: \' + member)\n    return raw\n\n\ndef phash_image(image):\n    a = np.asarray(image.convert(\'L\').resize((32, 32), Image.Resampling.LANCZOS), dtype=float)\n    low = dctn(a, norm=\'ortho\')[:8, :8]\n    bits = low > np.median(low.flatten()[1:]); bits[0, 0] = False\n    return int.from_bytes(np.packbits(bits.flatten()).tobytes(), \'big\')\n\n\nclass NearIndex:\n    def __init__(self):\n        self.tables = [defaultdict(list) for _ in range(8)]\n\n    def add(self, value, iid):\n        for b in range(8):\n            self.tables[b][(value >> (8*b)) & 255].append((value, iid))\n\n    def find(self, value, max_distance=4, limit=5):\n        found, seen = [], set()\n        for b in range(8):\n            for prior, iid in self.tables[b].get((value >> (8*b)) & 255, []):\n                if iid in seen: continue\n                seen.add(iid); distance = (prior ^ value).bit_count()\n                if distance <= max_distance: found.append((distance, iid))\n        return sorted(found)[:limit]\n\n\ndef iou_matrix(a, b):\n    if not a or not b: return np.zeros((len(a), len(b)))\n    a = np.asarray([r[\'bbox\'] for r in a], float); b = np.asarray([r[\'bbox\'] for r in b], float)\n    low = np.maximum(a[:, None, :2], b[None, :, :2]); high = np.minimum(a[:, None, 2:], b[None, :, 2:])\n    inter = np.maximum(high-low, 0).prod(2)\n    area_a = np.maximum(a[:, 2:]-a[:, :2], 0).prod(1)\n    area_b = np.maximum(b[:, 2:]-b[:, :2], 0).prod(1)\n    return inter / np.maximum(area_a[:, None] + area_b[None, :] - inter, 1e-12)\n\n\ndef parse_labels(raw, width, height):\n    """Registra todas as linhas; nenhuma correção silenciosa dos rótulos."""\n    boxes, issues = [], []\n    try: text = raw.decode(\'utf-8-sig\')\n    except UnicodeDecodeError:\n        return [], [{\'line\': 0, \'code\': \'label_nao_utf8\', \'blocking\': True}]\n    for ln, line in enumerate(text.splitlines(), 1):\n        if not line.strip(): continue\n        base = {\'line\': ln, \'raw\': line[:180]}\n        try:\n            vals = [float(x) for x in line.split()]\n            if len(vals) != 5 or not all(math.isfinite(x) for x in vals): raise ValueError()\n            cid, x, y, w, h = vals\n            if not cid.is_integer() or not 0 <= cid < len(NAMES): raise ValueError()\n        except ValueError:\n            issues.append({**base, \'code\': \'linha_ou_classe_invalida\', \'blocking\': True}); continue\n        norm = [x-w/2, y-h/2, x+w/2, y+h/2]\n        if min(w, h) <= 0 or min(norm) < -1e-6 or max(norm) > 1+1e-6:\n            issues.append({**base, \'code\': \'caixa_fora_ou_area_invalida\', \'blocking\': True}); continue\n        box = [norm[0]*width, norm[1]*height, norm[2]*width, norm[3]*height]\n        side = min(w*width, h*height)*640/max(width, height)\n        flags = []\n        if side < 16: flags.append(\'menor_lado_em_640_inferior_a_16\')\n        if max(w*width/(h*height), h*height/(w*width)) > 12: flags.append(\'proporcao_extrema_maior_12\')\n        if min(norm) < 0 or max(norm) > 1: flags.append(\'arredondamento_na_borda_ate_1e6\')\n        obj = {\'index\': len(boxes), \'line\': ln, \'classe_id\': int(cid), \'classe\': NAMES[int(cid)],\n               \'bbox_norm\': norm, \'bbox\': box, \'area_relativa\': w*h,\n               \'menor_lado_em_640\': side, \'flags\': flags}\n        boxes.append(obj)\n        for flag in flags: issues.append({**base, \'code\': flag, \'blocking\': False})\n    matrix = iou_matrix(boxes, boxes)\n    for i in range(len(boxes)):\n        for j in range(i+1, len(boxes)):\n            same = boxes[i][\'classe_id\'] == boxes[j][\'classe_id\']\n            if matrix[i, j] >= (.95 if same else .90):\n                code = \'caixas_mesma_classe_quase_duplicadas\' if same else \'classes_diferentes_sobreposicao_alta\'\n                issues.append({\'line\': boxes[i][\'line\'], \'other_line\': boxes[j][\'line\'],\n                               \'code\': code, \'iou\': float(matrix[i,j]), \'blocking\': False})\n    return boxes, issues\n\n\ndef validate_coverage(cov, known, gt, source):\n    if source == \'home\': return []\n    problems = []\n    pos = cov.get(\'classes_positivas_verificadas_exaustivas\', [])\n    neg = cov.get(\'classes_negativas_verificadas\', [])\n    ids = pos + neg\n    if (any(type(c) is not int or not 0 <= c < 12 for c in ids)\n            or type(known) is not int or not 0 <= known <= ALL_KNOWN):\n        return [{\'line\': 0, \'code\': \'mascara_cobertura_invalida\', \'blocking\': True}]\n    if (set(pos) & set(neg) or sum(1 << c for c in set(ids)) != known\n            or cov.get(\'known_mask\') != known):\n        problems.append({\'line\': 0, \'code\': \'cobertura_inconsistente\', \'blocking\': True})\n    present = {o[\'classe_id\'] for o in gt}\n    if present & set(neg): problems.append({\'line\': 0, \'code\': \'caixa_em_classe_negativa_verificada\', \'blocking\': True})\n    if set(pos) - present: problems.append({\'line\': 0, \'code\': \'classe_positiva_exaustiva_sem_caixa\', \'blocking\': True})\n    return problems\n\n\ndef load_history(csv_path, json_path, expected):\n    split = json.loads(Path(json_path).read_text()); claimed = split.pop(\'divisao_sha256\')\n    if claimed != expected or digest(split) != claimed: raise ValueError(\'Divisão histórica alterada.\')\n    with Path(csv_path).open(encoding=\'utf-8-sig\', newline=\'\') as f: rows = list(csv.DictReader(f))\n    if (len(rows) != 100000 or len({r[\'id\'] for r in rows}) != 100000\n            or Counter(r[\'split\'] for r in rows) != {\'train\': 80000, \'test\': 20000}):\n        raise ValueError(\'Divisão diferente dos 100 mil IDs históricos.\')\n    for role, key in [(\'train\', \'ids_treino\'), (\'test\', \'ids_teste\')]:\n        if {r[\'id\'] for r in rows if r[\'split\'] == role} != set(split[key]):\n            raise ValueError(\'CSV/JSON históricos divergem.\')\n    if len({r[\'imagem_sha256\'] for r in rows}) != len(rows): raise ValueError(\'Histórico com bytes repetidos.\')\n    return rows\n\n\ndef history_distribution(rows, sample_ids):\n    result = []\n    for label, cohort in [(\'train_historico_80000\', [r for r in rows if r[\'split\']==\'train\']),\n                          (\'recorte_cinco_lotes\', [r for r in rows if r[\'id\'] in sample_ids])]:\n        for cid, name in enumerate(NAMES):\n            result.append({\'coorte\': label, \'classe_id\': cid, \'classe\': name, \'imagens\': len(cohort),\n                           \'classe_primaria_imagens\': sum(int(r[\'classe_primaria\']) == cid for r in cohort),\n                           \'classe_com_cobertura_imagens\': sum(bool(int(r[\'known_mask\']) & (1 << cid)) for r in cohort)})\n    return result\n\n\ndef home_catalog(zip_path, expected):\n    if sha_file(zip_path) != expected[\'home_zip\']: raise ValueError(\'ZIP Home alterado.\')\n    with zipfile.ZipFile(zip_path) as z:\n        inspect_zip(z); manifest = json.loads(z.read(\'verificacao_preparacao.json\'))\n    if (manifest[\'test\'][\'sha256\'] != expected[\'test_home\'] or manifest[\'test\'][\'imagens\'] != 403\n            or {int(k): v for k,v in manifest[\'classes\'].items()} != dict(enumerate(NAMES))):\n        raise ValueError(\'Manifesto Home incompatível.\')\n    root = manifest[\'dataset_dir\']; items = {r[\'arquivo\']: r for r in manifest[\'arquivos\']}\n    if len(items) != len(manifest[\'arquivos\']): raise ValueError(\'Manifesto Home repetido.\')\n    images = {name: r for name,r in items.items() if name.startswith(root+\'/images/\')\n              and Path(name).suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.bmp\',\'.webp\'}}\n    train = {n:r for n,r in images.items() if n.startswith(root+\'/images/train/raw_train/\')}\n    val = {n:r for n,r in images.items() if n.startswith(root+\'/images/val/raw_train/\')}\n    if len(train)!=1824 or len(val)!=456: raise ValueError(\'Quantidades Home incompatíveis.\')\n    protected = [(r[\'sha256\'], \'home_protegido:\'+n) for n,r in images.items() if n not in train]\n    catalog = []\n    for name,item in sorted(train.items()):\n        label = name.replace(\'/images/\',\'/labels/\').rsplit(\'.\',1)[0]+\'.txt\'\n        if label not in items: raise ValueError(\'Label Home ausente do manifesto.\')\n        catalog.append({\'id\':\'home:\'+name, \'source\':\'home\', \'role\':\'train\', \'image_member\':name,\n                        \'label_member\':label, \'sha256\':item[\'sha256\'], \'label_sha256\':items[label][\'sha256\'],\n                        \'known_mask\':ALL_KNOWN, \'lote\':\'homeobjects_sunrgbd_validado.zip\'})\n    return catalog, protected\n\n\ndef protected_index(history, home_protected, new_manifest):\n    by_sha, ids, near = defaultdict(list), set(), NearIndex()\n    for sha, iid in home_protected: by_sha[sha].append(iid)\n    for r in history:\n        if r[\'split\']==\'test\': by_sha[r[\'imagem_sha256\']].append(\'oi_test:\'+r[\'id\']); ids.add(r[\'id\'])\n    for r in new_manifest[\'arquivos\']:\n        by_sha[r[\'sha256\']].append(r[\'role\']+\':\'+r[\'id\']); ids.add(r[\'id\'])\n        if r.get(\'phash\') is not None:\n            near.add(int(r[\'phash\'],16), r[\'role\']+\':\'+r[\'id\'])\n    return by_sha, ids, near\n\n\ndef materialize_one(z, spec, output, protected_sha, protected_ids, protected_near):\n    if spec[\'role\'] != \'train\': raise ValueError(\'Papel proibido na auditoria de treino.\')\n    iid = spec.get(\'oi_id\', spec[\'id\'])\n    hit = protected_sha.get(spec[\'sha256\'], [])\n    record = {**spec, \'gt\': [], \'issues\': [], \'protected_matches\': list(hit), \'image\': None}\n    if hit or iid in protected_ids:\n        record[\'issues\'].append({\'line\':0, \'code\':\'coincidencia_com_conjunto_protegido\', \'blocking\':True})\n        record[\'status_input\']=\'quarentena_sem_extracao\'; return record\n    raw_label = checked_read(z, spec[\'label_member\'], spec[\'label_sha256\'])\n    cov = {}\n    if spec.get(\'coverage_member\'):\n        cov = json.loads(checked_read(z, spec[\'coverage_member\'], spec[\'coverage_sha256\']))\n    raw = checked_read(z, spec[\'image_member\'], spec[\'sha256\'])\n    record[\'label_text\'] = raw_label.decode(\'utf-8-sig\', errors=\'replace\'); record[\'coverage_original\']=cov\n    try:\n        with Image.open(io.BytesIO(raw)) as im:\n            width,height = im.size; im.verify()\n        with Image.open(io.BytesIO(raw)) as im:\n            im.load(); ph = phash_image(im)\n            orientation = im.getexif().get(274,1)\n        if spec.get(\'width\') is not None and (spec[\'width\'],spec[\'height\']) != (width,height):\n            record[\'issues\'].append({\'line\':0,\'code\':\'dimensoes_divergem_do_manifesto\',\'blocking\':True})\n        record.update(width=width,height=height,phash=f\'{ph:016x}\',exif_orientation=int(orientation))\n        hits = protected_near.find(ph)\n        record[\'protected_phash_matches\'] = [{\'distancia\':d,\'id\':n} for d,n in hits]\n        if hits: record[\'issues\'].append({\'line\':0,\'code\':\'semelhanca_phash_com_protegido\',\'blocking\':False})\n        if orientation != 1:\n            record[\'issues\'].append({\'line\':0,\'code\':\'orientacao_exif_nao_padrao\',\'blocking\':True})\n        gt, issues = parse_labels(raw_label,width,height)\n        record[\'gt\']=gt; record[\'issues\'] += issues + validate_coverage(cov,spec[\'known_mask\'],gt,spec[\'source\'])\n        if not gt: record[\'issues\'].append({\'line\':0,\'code\':\'imagem_sem_caixas\',\'blocking\':False})\n        # Não inferir pixels de uma possível cópia perceptual da reserva/validação.\n        if hits: record[\'status_input\']=\'quarentena_phash_sem_inferencia\'; return record\n        image = Path(output)/\'images\'/spec[\'source\']/(hashlib.sha256(spec[\'id\'].encode()).hexdigest()+\'.jpg\')\n        image.parent.mkdir(parents=True,exist_ok=True); image.write_bytes(raw)\n        record[\'image\']=str(image)\n        record[\'status_input\']=\'bloqueio_tecnico\' if any(i[\'blocking\'] for i in record[\'issues\']) else \'legivel\'\n    except (OSError,ValueError,Image.DecompressionBombError) as exc:\n        record[\'issues\'].append({\'line\':0,\'code\':\'imagem_ilegivel\',\'blocking\':True,\'detail\':str(exc)[:160]})\n        record[\'status_input\']=\'bloqueio_tecnico\'\n    return record\n\n\ndef extract_home(zip_path,catalog,output,protected,progress=None):\n    records=[]\n    with zipfile.ZipFile(zip_path) as z:\n        inspect_zip(z)\n        for i,spec in enumerate(catalog,1):\n            records.append(materialize_one(z,spec,output,*protected))\n            if progress and (i%200==0 or i==len(catalog)): progress(f\'Home: {i}/{len(catalog)} imagens auditadas\')\n    return records\n\n\ndef extract_oi_package(zip_path,item,history_map,expected_cfg,output,protected):\n    if sha_file(zip_path)!=item[\'zip_sha256\'] or Path(zip_path).stat().st_size!=item[\'zip_bytes\']:\n        raise ValueError(\'Lote incompleto ou diferente do histórico: \'+item[\'arquivo\'])\n    records=[]\n    with zipfile.ZipFile(zip_path) as z:\n        inspect_zip(z); m=json.loads(z.read(\'manifesto_lote.json\'))\n        if m[\'config_sha256\']!=expected_cfg or m[\'origem_split\']!=\'train\': raise ValueError(\'Lote fora do TRAIN esperado.\')\n        accepted=m[\'aceitas\']\n        expected_ids={r[\'id\'] for r in history_map.values() if r[\'lote\']==item[\'arquivo\']}\n        if len(accepted)!=len(expected_ids) or {r[\'id\'] for r in accepted}!=expected_ids:\n            raise ValueError(\'IDs do lote divergentes da divisão histórica.\')\n        for r in sorted(accepted,key=lambda r:r[\'id\']):\n            h=history_map[r[\'id\']]\n            if (h[\'split\']!=\'train\' or h[\'lote\']!=item[\'arquivo\'] or h[\'imagem_sha256\']!=r[\'sha256\']\n                    or int(h[\'known_mask\'])!=r[\'known_mask\']): raise ValueError(\'Identidade de treino incompatível.\')\n            spec={\'id\':\'oi:\'+r[\'id\'],\'oi_id\':r[\'id\'],\'source\':\'openimages\',\'role\':\'train\',\n                  \'image_member\':r[\'imagem\'],\'label_member\':r[\'label\'],\'coverage_member\':r[\'cobertura\'],\n                  \'sha256\':r[\'sha256\'],\'label_sha256\':r[\'label_sha256\'],\'coverage_sha256\':r[\'cobertura_sha256\'],\n                  \'known_mask\':r[\'known_mask\'],\'width\':r[\'width\'],\'height\':r[\'height\'],\n                  \'lote\':item[\'arquivo\'],\'classe_primaria\':int(h[\'classe_primaria\'])}\n            records.append(materialize_one(z,spec,output,*protected))\n    return records\n\n\ndef duplicate_rows(records):\n    exact, rows, near = defaultdict(list), [], NearIndex()\n    for r in sorted(records,key=lambda r:r[\'id\']):\n        for prior in exact[r[\'sha256\']]:\n            rows.append({\'id\':r[\'id\'],\'similar_id\':prior,\'type\':\'bytes_iguais\',\'distance\':0})\n        exact[r[\'sha256\']].append(r[\'id\'])\n        if r.get(\'phash\') is not None:\n            value=int(r[\'phash\'],16)\n            for distance,prior in near.find(value):\n                rows.append({\'id\':r[\'id\'],\'similar_id\':prior,\'type\':\'semelhanca_phash\',\'distance\':distance})\n            near.add(value,r[\'id\'])\n    return rows\n\n\ndef manifest_public(records):\n    return [{k:v for k,v in r.items() if k!=\'image\'} for r in records]\n\n\ndef assert_inputs_unchanged(records):\n    for r in records:\n        if r.get(\'image\') and sha_file(r[\'image\'])!=r[\'sha256\']:\n            raise ValueError(\'Imagem de treino mudou: \'+r[\'id\'])\n\n\ndef eligible(records):\n    return [r for r in records if r[\'role\']==\'train\' and r[\'status_input\']==\'legivel\' and r.get(\'image\')]\n\n\ndef chunks(records,n=100):\n    for source in sorted({r[\'source\'] for r in records}):\n        group=sorted([r for r in records if r[\'source\']==source],key=lambda r:r[\'id\'])\n        for i in range(0,len(group),n): yield source+f\'_{i//n:04d}\',group[i:i+n]\n\n\ndef record_identity(r):\n    return {k:r.get(k) for k in [\'id\',\'role\',\'source\',\'sha256\',\'label_sha256\',\'coverage_sha256\',\'known_mask\',\'width\',\'height\']}\n\n\ndef cache_read(path,signature,ids,validator=None):\n    path=Path(path)\n    if not path.is_file(): return None\n    obj=load_gz(path)\n    if obj.get(\'signature\')!=signature or [r[\'id\'] for r in obj.get(\'rows\',[])]!=ids:\n        raise ValueError(\'Cache incompatível: \'+str(path))\n    if obj.get(\'rows_sha256\')!=digest(obj[\'rows\']):\n        raise ValueError(\'Conteúdo do cache alterado: \'+str(path))\n    if validator:\n        for r in obj[\'rows\']: validator(r)\n    return obj[\'rows\']\n\n\ndef cache_write(path,signature,rows):\n    save_gz(path,{\'signature\':signature,\'rows_sha256\':digest(rows),\'rows\':rows})\n\n\ndef recoverable_cache_read(path,signature,ids,validator=None):\n    try:return cache_read(path,signature,ids,validator)\n    except (ValueError,KeyError,TypeError,EOFError,OSError) as exc:\n        path=Path(path)\n        rejected=path.parent/\'rejeitados\';rejected.mkdir(exist_ok=True)\n        target=rejected/(path.name+\'.\'+str(time.time_ns())+\'.rejeitado\')\n        if path.exists():path.replace(target)\n        save_json(str(target)+\'.json\',{\'cache\':path.name,\'motivo\':str(exc)[:240],\'recalculado_automaticamente\':True})\n        print(\'Cache rejeitado; recalculando automaticamente:\',path.name,flush=True)\n        return None\n\n\ndef package(output):\n    output=Path(output); archive=output/\'pacote_auditoria_15.zip\'; tmp=archive.with_suffix(\'.zip.tmp\')\n    with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_DEFLATED,compresslevel=4) as z:\n        for p in sorted(output.rglob(\'*\')):\n            if p.is_file() and p not in {archive,tmp} and not p.name.endswith(\'.tmp\'):\n                z.write(p,p.relative_to(output))\n    tmp.replace(archive)\n    return archive\n', 'audit15_models.py': '"""YOLO e CLIP são sinais de triagem em TRAIN, não validadores de verdade."""\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nimport audit15_core as core\n\nCLIP_COMMIT = \'d05afc436d78f1c48dc0dbf8e5980a9d471f35f6\'\nCLIP_SHA = \'40d365715913c9da98579312b702a82c18be219cc2a73407c4526f58eba950af\'\nCONCEPTS = [\n    (\'bed\', \'a bed with a mattress\'),\n    (\'sofa\', \'a sofa or couch\'),\n    (\'chair\', \'a chair, stool or armchair for one person\'),\n    (\'table\', \'a table, dining table, coffee table or desk\'),\n    (\'lamp\', \'a lamp or indoor light fixture\'),\n    (\'tv\', \'a television screen\'),\n    (\'laptop\', \'a laptop computer\'),\n    (\'wardrobe\', \'a wardrobe or tall closet for storing clothes\'),\n    (\'window\', \'a window in a building\'),\n    (\'door\', \'a door in a building\'),\n    (\'potted plant\', \'a real potted plant\'),\n    (\'photo frame\', \'a picture frame or framed photograph\'),\n    (\'generic cabinet\', \'a generic storage cabinet or cupboard\'),\n    (\'bookcase\', \'a bookcase or open shelving unit\'),\n    (\'tv stand\', \'a low TV stand or media console\'),\n    (\'chest of drawers\', \'a chest of drawers or dresser\'),\n    (\'sideboard\', \'a sideboard or buffet cabinet\'),\n    (\'refrigerator\', \'a refrigerator\'),\n    (\'radiator\', \'a radiator or space heater\'),\n    (\'outdoor vegetation\', \'outdoor trees, bushes or grass\'),\n    (\'person\', \'a person\'),\n    (\'wall or floor\', \'an empty wall or floor\'),\n    (\'decorative object\', \'a vase, ornament or decorative object\'),\n    (\'unrecognizable detail\', \'an unrecognizable close-up detail or texture\'),\n]\nTEMPLATES = [\'a photo of {}.\', \'a cropped photo of {}.\', \'an image showing {}.\']\nSCENES = [\n    (\'interior\', \'a photo of the inside of a furnished room\'),\n    (\'exterior\', \'a photo of an outdoor scene, street or garden\'),\n    (\'produto_fundo_liso\', \'a studio product photograph on a plain background\'),\n    (\'colagem_ou_desenho\', \'a collage, illustration or drawing\'),\n    (\'detalhe_sem_contexto\', \'a close-up photograph with no visible room context\'),\n]\nPREDICT_ARGS = {\'imgsz\':640,\'conf\':.15,\'iou\':.70,\'max_det\':300,\n                \'half\':False,\'augment\':False,\'rect\':True,\'verbose\':False}\nSEMANTIC_RULE = {\'views\':[1.0,1.2],\'min_crop_side\':8,\'min_cosine\':.20,\n                 \'min_margin_over_label_both_views\':.03,\n                 \'unmatched_focus_min_score\':.25,\'unmatched_per_image_limit\':10,\n                 \'scene_min_cosine\':.20,\'scene_margin\':.02,\n                 \'thresholds_calibrated\':False}\n\n\ndef environment():\n    import torch, torchvision, ultralytics, PIL, scipy\n    return {\'torch\':torch.__version__,\'torchvision\':torchvision.__version__,\n            \'ultralytics\':ultralytics.__version__,\'numpy\':np.__version__,\n            \'pillow\':PIL.__version__,\'scipy\':scipy.__version__,\n            \'ftfy\':importlib.metadata.version(\'ftfy\'),\'regex\':importlib.metadata.version(\'regex\'),\n            \'clip_commit\':CLIP_COMMIT,\'gpu\':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}\n\n\ndef validate_prediction_row(row):\n    for p in row[\'predictions\']:\n        c, score, box = p[\'classe_id\'], p[\'confianca\'], p[\'bbox\']\n        if (type(c) is not int or not 0 <= c < 12 or not isinstance(score,(int,float))\n                or not math.isfinite(score) or not 0 <= score <= 1 or len(box)!=4\n                or not all(isinstance(x,(int,float)) and math.isfinite(x) for x in box)\n                or box[2]<box[0] or box[3]<box[1]):\n            raise ValueError(\'Previsão estruturalmente inválida: \'+row[\'id\'])\n\n\ndef positive_area(p):\n    b=p[\'bbox\']; return b[2]>b[0] and b[3]>b[1]\n\n\ndef train_guard(records):\n    if any(r.get(\'role\')!=\'train\' or r.get(\'status_input\')!=\'legivel\' or not r.get(\'image\') for r in records):\n        raise ValueError(\'Somente treino legível e sem quarentena pode entrar nos modelos.\')\n\n\ndef matches(gt,pred,threshold=.5):\n    matrix=core.iou_matrix(gt,pred)\n    options=[]\n    for gi,g in enumerate(gt):\n        for pi,p in enumerate(pred):\n            if g[\'classe_id\']==p[\'classe_id\'] and matrix[gi,pi]>=threshold:\n                options.append((-float(matrix[gi,pi]),gi,pi))\n    pairs=[]; used_g=set(); used_p=set()\n    for neg,gi,pi in sorted(options):\n        if gi not in used_g and pi not in used_p:\n            used_g.add(gi);used_p.add(pi);pairs.append((gi,pi,-neg))\n    return pairs, [i for i in range(len(gt)) if i not in used_g], [i for i in range(len(pred)) if i not in used_p]\n\n\ndef detector_evidence(record,predictions):\n    # Todas as classes anotadas são associadas; cobertura restringe a interpretação das sobras.\n    pred=[p for p in predictions if p[\'confianca\']>=.25 and positive_area(p)]\n    pairs,missing,extras=matches(record[\'gt\'],pred)\n    gmatch={gi:(pi,overlap) for gi,pi,overlap in pairs}\n    gt_rows=[]\n    for gi,g in enumerate(record[\'gt\']):\n        row={\'gt_index\':gi,\'matched_at_025\':gi in gmatch,\'matched_score\':None,\'iou\':None}\n        if gi in gmatch:\n            pi,overlap=gmatch[gi];row.update(matched_score=pred[pi][\'confianca\'],iou=overlap)\n        gt_rows.append(row)\n    extra_rows=[]\n    for pi in extras:\n        p=pred[pi]; known=bool(record[\'known_mask\'] & (1 << p[\'classe_id\']))\n        overlap=core.iou_matrix(record[\'gt\'],[p])[:,0]\n        best=int(np.argmax(overlap)) if len(overlap) else None\n        extra_rows.append({**p,\'known_class\':known,\'best_iou_any_label\':float(overlap[best]) if best is not None else 0.,\n                           \'overlapping_label_class\':record[\'gt\'][best][\'classe_id\'] if best is not None else None,\n                           \'interpretation\':\'sobra_em_classe_coberta_a_investigar\' if known else \'cobertura_desconhecida_nao_contar_como_erro\'})\n    selected=sorted([p for p in extra_rows if p[\'classe_id\'] in core.FOCUS],\n                    key=lambda p:(-p[\'confianca\'],p[\'classe_id\'],p[\'bbox\']))[:SEMANTIC_RULE[\'unmatched_per_image_limit\']]\n    return {\'gt\':gt_rows,\'extras\':extra_rows,\'extra_crops\':selected,\n            \'gt_without_match_at_025\':len(missing),\n            \'zero_area_proposals\':sum(not positive_area(p) for p in predictions),\n            \'saturated_max_det\':len(predictions)>=PREDICT_ARGS[\'max_det\']}\n\n\ndef infer_yolo(records,weight,expected_sha,cache_dir,context,device,checkpoint=None,predict_fn=None):\n    train_guard(records)\n    if core.sha_file(weight)!=expected_sha: raise ValueError(\'Peso B alterado antes da inferência.\')\n    model=None\n    if predict_fn is None:\n        from ultralytics import YOLO\n        model=YOLO(str(weight))\n        names=model.names\n        if isinstance(names,list): names=dict(enumerate(names))\n        if {int(k):v for k,v in names.items()}!=dict(enumerate(core.NAMES)): raise ValueError(\'Classes dos pesos incompatíveis.\')\n        def predict_fn(r):\n            result=model.predict(source=r[\'image\'],device=device,**PREDICT_ARGS)[0]\n            return [{\'classe_id\':int(c),\'confianca\':float(s),\'bbox\':[float(x) for x in b]}\n                    for b,c,s in zip(result.boxes.xyxy.cpu().tolist(),result.boxes.cls.cpu().tolist(),result.boxes.conf.cpu().tolist())]\n    results={};done=0;fresh=0;t0=time.monotonic()\n    for name,group in core.chunks(records):\n        sig=core.digest({\'context\':context,\'weight\':expected_sha,\'predict\':PREDICT_ARGS,\n                         \'records\':[core.record_identity(r) for r in group]})\n        path=Path(cache_dir)/\'yolo\'/(name+\'.json.gz\')\n        rows=core.recoverable_cache_read(path,sig,[r[\'id\'] for r in group],validate_prediction_row)\n        if rows is None:\n            rows=[]\n            for r in group:\n                row={\'id\':r[\'id\'],\'predictions\':predict_fn(r)};validate_prediction_row(row); rows.append(row);fresh+=1\n            core.cache_write(path,sig,rows)\n        results.update({r[\'id\']:r[\'predictions\'] for r in rows});done+=len(group)\n        elapsed=time.monotonic()-t0;rate=fresh/max(elapsed,1e-9)\n        eta=(len(records)-done)/rate/60 if rate else None\n        message=f\'YOLO: {done}/{len(records)}; novas inferências nesta passagem: {fresh}\'\n        if eta is not None:message+=f\'; estimativa restante pela velocidade atual: {eta:.1f} min\'\n        print(message,flush=True)\n        if checkpoint: checkpoint(\'cache YOLO \'+name)\n    del model\n    return results,{\'images\':len(records),\'new_inferences\':fresh,\'seconds\':time.monotonic()-t0}\n\n\ndef crop_square(image,box,factor=1.):\n    w,h=image.size;x1,y1,x2,y2=box\n    cx=(x1+x2)/2;cy=(y1+y2)/2;bw=(x2-x1)*factor;bh=(y2-y1)*factor\n    xy=[max(0,math.floor(cx-bw/2)),max(0,math.floor(cy-bh/2)),\n        min(w,math.ceil(cx+bw/2)),min(h,math.ceil(cy+bh/2))]\n    if xy[2]<=xy[0] or xy[3]<=xy[1]: return None\n    region=image.crop(tuple(xy)).convert(\'RGB\'); side=max(region.size)\n    square=Image.new(\'RGB\',(side,side),(127,127,127));square.paste(region,((side-region.width)//2,(side-region.height)//2))\n    return square.resize((224,224),Image.Resampling.BICUBIC)\n\n\ndef semantic_result(scores,expected):\n    a=np.asarray(scores,dtype=float)\n    if a.shape!=(2,len(CONCEPTS)) or not np.isfinite(a).all() or np.any(np.abs(a)>1.0001):\n        raise ValueError(\'Scores CLIP incompatíveis.\')\n    best=np.argmax(a,axis=1);top=[float(a[i,best[i]]) for i in range(2)]\n    delta=[float(a[i,best[i]]-a[i,expected]) for i in range(2)]\n    stable=bool(best[0]==best[1]); agree=bool(np.all(best==expected))\n    strong=bool(stable and not agree and min(top)>=SEMANTIC_RULE[\'min_cosine\']\n                and min(delta)>=SEMANTIC_RULE[\'min_margin_over_label_both_views\'])\n    status=\'divergencia_forte_ia\' if strong else (\'compativel_ia\' if agree else \'inconclusivo_ia\')\n    return {\'status\':status,\'expected\':core.NAMES[expected],\n            \'top_tight\':CONCEPTS[int(best[0])][0],\'top_context\':CONCEPTS[int(best[1])][0],\n            \'top_cosine_tight\':top[0],\'top_cosine_context\':top[1],\n            \'margin_over_label_tight\':delta[0],\'margin_over_label_context\':delta[1],\n            \'scores\':a.tolist(),\'probability_calibrated\':False}\n\n\ndef scene_result(scores):\n    a=np.asarray(scores,dtype=float)\n    if a.shape!=(len(SCENES),) or not np.isfinite(a).all() or np.any(np.abs(a)>1.0001):\n        raise ValueError(\'Scores de domínio inválidos.\')\n    idx=np.argsort(-a,kind=\'stable\');top=int(idx[0]);margin=float(a[top]-a[idx[1]])\n    indicative=bool(a[top]>=SEMANTIC_RULE[\'scene_min_cosine\'] and margin>=SEMANTIC_RULE[\'scene_margin\'])\n    return {\'top\':SCENES[top][0],\'cosine\':float(a[top]),\'margin\':margin,\n            \'indicative\':indicative,\'scores\':a.tolist(),\'probability_calibrated\':False}\n\n\nclass ClipScorer:\n    def __init__(self,device,download_root):\n        import clip, torch\n        import clip.clip as impl\n        expected_url=impl._MODELS[\'ViT-B/32\']\n        if expected_url.split(\'/\')[-2]!=CLIP_SHA: raise ValueError(\'Peso CLIP não corresponde ao hash congelado.\')\n        self.device=\'cuda:0\' if device==0 else \'cpu\';self.torch=torch;self.clip=clip\n        self.model,self.preprocess=clip.load(\'ViT-B/32\',device=self.device,jit=False,download_root=str(download_root))\n        path=Path(download_root)/\'ViT-B-32.pt\'\n        if core.sha_file(path)!=CLIP_SHA: raise ValueError(\'Hash do CLIP inválido.\')\n        self.model=self.model.float().eval()\n        self.model.requires_grad_(False)\n        self.batch_size=32 if device==0 else 8\n        self.concepts=self._text_features([p for _,p in CONCEPTS])\n        self.scenes=self._text_features([p for _,p in SCENES],templates=[\'{}\'])\n\n    def _text_features(self,descriptions,templates=TEMPLATES):\n        torch=self.torch\n        sentences=[t.format(d) for d in descriptions for t in templates]\n        pieces=[]\n        with torch.inference_mode():\n            for i in range(0,len(sentences),32):\n                x=self.model.encode_text(self.clip.tokenize(sentences[i:i+32]).to(self.device)).float()\n                pieces.append(x/x.norm(dim=-1,keepdim=True).clamp_min(1e-12))\n            x=torch.cat(pieces).reshape(len(descriptions),len(templates),-1).mean(dim=1)\n            return x/x.norm(dim=-1,keepdim=True).clamp_min(1e-12)\n\n    def score(self,images,scenes=False):\n        torch=self.torch;features=[]\n        if not images:return np.empty((0,len(SCENES) if scenes else len(CONCEPTS)))\n        with torch.inference_mode():\n            for i in range(0,len(images),self.batch_size):\n                inp=torch.stack([self.preprocess(im) for im in images[i:i+self.batch_size]]).to(self.device)\n                x=self.model.encode_image(inp).float();x=x/x.norm(dim=-1,keepdim=True).clamp_min(1e-12)\n                features.append((x @ (self.scenes if scenes else self.concepts).T).cpu().numpy())\n        out=np.concatenate(features)\n        if not np.isfinite(out).all():raise ValueError(\'CLIP retornou valores não finitos.\')\n        return out\n\n\ndef score_record(record,predictions,scorer):\n    evidence=detector_evidence(record,predictions)\n    jobs=[(\'gt\',g[\'index\'],g) for g in record[\'gt\']]\n    jobs += [(\'extra\',i,p) for i,p in enumerate(evidence[\'extra_crops\'])]\n    rendered=[];positions=[];details=[]\n    with Image.open(record[\'image\']) as im:\n        image=im.convert(\'RGB\')\n        # A cena inteira é reduzida com padding, preservando os cantos da imagem.\n        scene_img=crop_square(image,[0,0,*image.size])\n        scene=scene_result(scorer.score([scene_img],scenes=True)[0])\n        for kind,index,obj in jobs:\n            box=obj[\'bbox\'];entry={\'kind\':kind,\'index\':index,\'classe_id\':obj[\'classe_id\'],\'bbox\':box}\n            if min(box[2]-box[0],box[3]-box[1])<SEMANTIC_RULE[\'min_crop_side\']:\n                details.append({**entry,\'status\':\'recorte_pequeno_sem_analise\',\'scores\':None});continue\n            crops=[crop_square(image,box,factor) for factor in SEMANTIC_RULE[\'views\']]\n            if any(c is None for c in crops):\n                details.append({**entry,\'status\':\'recorte_invalido_sem_analise\',\'scores\':None});continue\n            positions.append((entry,len(rendered)));rendered.extend(crops)\n        # Inferência em minibatches dentro de score; uma imagem é aberta uma só vez.\n        scores=scorer.score(rendered)\n        for entry,index in positions:details.append({**entry,**semantic_result(scores[index:index+2],entry[\'classe_id\'])})\n    return {\'id\':record[\'id\'],\'scene\':scene,\'objects\':sorted(details,key=lambda x:(x[\'kind\'],x[\'index\'])),\n            \'detector\':evidence}\n\n\ndef validate_semantic_row(row):\n    scene_result(row[\'scene\'][\'scores\'])\n    for obj in row[\'objects\']:\n        if obj.get(\'scores\') is not None:\n            recalculated=semantic_result(obj[\'scores\'],obj[\'classe_id\'])\n            if recalculated[\'status\']!=obj[\'status\']:raise ValueError(\'Decisão CLIP incompatível com scores no cache.\')\n\n\ndef infer_semantics(records,predictions,cache_dir,context,device,model_dir,checkpoint=None,scorer=None):\n    train_guard(records);results={};fresh=0;done=0;start=time.monotonic()\n    for name,group in core.chunks(records):\n        sig=core.digest({\'context\':context,\'clip_sha\':CLIP_SHA,\'concepts\':CONCEPTS,\'templates\':TEMPLATES,\n                         \'scenes\':SCENES,\'rule\':SEMANTIC_RULE,\'records\':[core.record_identity(r) for r in group],\n                         \'predictions_sha256\':core.digest({r[\'id\']:predictions[r[\'id\']] for r in group})})\n        path=Path(cache_dir)/\'clip\'/(name+\'.json.gz\')\n        rows=core.recoverable_cache_read(path,sig,[r[\'id\'] for r in group],validate_semantic_row)\n        if rows is None:\n            if scorer is None:scorer=ClipScorer(device,model_dir)\n            rows=[]\n            for r in group:\n                row=score_record(r,predictions[r[\'id\']],scorer);validate_semantic_row(row);rows.append(row);fresh+=1\n                if len(rows)%10==0:print(f\'CLIP: bloco {name}, {len(rows)}/{len(group)} imagens\',flush=True)\n            core.cache_write(path,sig,rows)\n        results.update({r[\'id\']:r for r in rows});done+=len(group)\n        speed=fresh/max(time.monotonic()-start,1e-9)\n        msg=f\'CLIP: {done}/{len(records)} imagens; novas análises nesta passagem: {fresh}\'\n        if speed:msg+=f\'; estimativa restante pela velocidade atual: {(len(records)-done)/speed/60:.1f} min\'\n        print(msg,flush=True)\n        if checkpoint:checkpoint(\'cache CLIP \'+name)\n    return results,{\'images\':len(records),\'new_analyses\':fresh,\'seconds\':time.monotonic()-start}\n', 'audit15_reports.py': '"""Relatórios descritivos da auditoria, preservando a origem dos sinais."""\nfrom __future__ import annotations\n\nimport html\nimport json\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nfrom PIL import Image, ImageDraw, ImageFont, ImageOps\n\nimport audit15_core as core\nimport audit15_models as models\n\n\ndef html_table(rows):\n    if not rows:return \'<p>Sem registros neste recorte.</p>\'\n    fields=list(rows[0]);out=\'<div class="scroll"><table><thead><tr>\'\n    out+=\'\'.join(\'<th>\'+html.escape(str(k))+\'</th>\' for k in fields)+\'</tr></thead><tbody>\'\n    for row in rows:out+=\'<tr>\'+\'\'.join(\'<td>\'+html.escape(str(row.get(k,\'\')))+\'</td>\' for k in fields)+\'</tr>\'\n    return out+\'</tbody></table></div>\'\n\n\ndef font(size):\n    for p in [\'/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf\',\'/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf\']:\n        if Path(p).exists():return ImageFont.truetype(p,size)\n    return ImageFont.load_default()\n\n\ndef render_case(record,semantic,path):\n    with Image.open(record[\'image\']) as f: image=f.convert(\'RGB\')\n    draw=ImageDraw.Draw(image);size=max(12,min(image.size)//45);ft=font(size)\n    evidence={r[\'index\']:r for r in semantic[\'objects\'] if r[\'kind\']==\'gt\'}\n    strong=[]\n    for g in record[\'gt\']:\n        sem=evidence.get(g[\'index\'],{});bad=sem.get(\'status\')==\'divergencia_forte_ia\'\n        color=\'#f45175\' if bad else \'#43c9a0\'\n        box=tuple(max(0,min(record[\'width\'] if i%2==0 else record[\'height\'],x)) for i,x in enumerate(g[\'bbox\']))\n        draw.rectangle(box,outline=color,width=max(2,size//6))\n        label=f"{g[\'line\']}: {g[\'classe\']}"\n        if bad:label+=\' | IA: \'+sem.get(\'top_tight\',\'?\');strong.append(g)\n        x,y=box[:2]\n        extent=draw.textbbox((0,0),label,font=ft,stroke_width=1)\n        x=min(x,max(0,image.width-(extent[2]-extent[0])-3))\n        draw.text((x,max(0,y-size-3)),label,fill=color,font=ft,stroke_width=1,stroke_fill=\'#111827\')\n    canvas=Image.new(\'RGB\',(1280,650),\'#0c1727\')\n    overview=ImageOps.contain(image,(630,580));canvas.paste(overview,((640-overview.width)//2,50+(580-overview.height)//2))\n    focus=strong[0] if strong else next((g for g in record[\'gt\'] if g[\'classe_id\'] in core.FOCUS),None)\n    if focus:\n        with Image.open(record[\'image\']) as f: crop=models.crop_square(f.convert(\'RGB\'),focus[\'bbox\'],1.2)\n        crop=ImageOps.contain(crop,(580,580));canvas.paste(crop,(660+(600-crop.width)//2,50+(580-crop.height)//2))\n    header=ImageDraw.Draw(canvas)\n    header.text((15,12),\'Rótulos originais: verde. Divergência forte da IA: rosa.\',fill=\'white\',font=font(18))\n    header.text((660,12),\'Recorte com contexto; sinal automático, sem correção.\',fill=\'white\',font=font(17))\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);canvas.save(path,quality=86)\n\n\ndef report(output,records,semantics,predictions,duplicates,metadata):\n    output=Path(output); image_rows=[];box_rows=[];extra_rows=[];scene_counts=Counter()\n    duplicate_ids={r[k] for r in duplicates for k in [\'id\',\'similar_id\']}\n    for r in records:\n        sem=semantics.get(r[\'id\']); signals=[i[\'code\'] for i in r[\'issues\']]\n        by_gt={o[\'index\']:o for o in sem[\'objects\'] if o[\'kind\']==\'gt\'} if sem else {}\n        det={o[\'gt_index\']:o for o in sem[\'detector\'][\'gt\']} if sem else {}\n        strong=0;uncertain=0;focus_difficult=0\n        for g in r[\'gt\']:\n            obj=by_gt.get(g[\'index\'],{});status=obj.get(\'status\',\'nao_analisado_por_bloqueio\')\n            strong+=status==\'divergencia_forte_ia\';uncertain+=status in {\'inconclusivo_ia\',\'recorte_pequeno_sem_analise\',\'recorte_invalido_sem_analise\'}\n            if g[\'classe_id\'] in core.FOCUS and det.get(g[\'index\'],{}).get(\'matched_at_025\') is False:focus_difficult+=1\n            row={\'id\':r[\'id\'],\'source\':r[\'source\'],\'line\':g[\'line\'],\'classe_id\':g[\'classe_id\'],\'classe\':g[\'classe\'],\n                 \'bbox_original\':json.dumps(g[\'bbox\']),\'area_relativa\':g[\'area_relativa\'],\'menor_lado_em_640\':g[\'menor_lado_em_640\'],\n                 \'cobertura_classe\':bool(r[\'known_mask\'] & (1 << g[\'classe_id\'])),\n                 \'sinais_geometria\':\'|\'.join(g[\'flags\']),\'triagem_ia\':status,\n                 \'top_ia_recorte\':obj.get(\'top_tight\',\'\'),\'top_ia_contexto\':obj.get(\'top_context\',\'\'),\n                 \'margem_vs_rotulo_recorte\':obj.get(\'margin_over_label_tight\'),\n                 \'margem_vs_rotulo_contexto\':obj.get(\'margin_over_label_context\'),\n                 \'correspondencia_B_025\':det.get(g[\'index\'],{}).get(\'matched_at_025\'),\n                 \'confianca_B_correspondente\':det.get(g[\'index\'],{}).get(\'matched_score\')}\n            box_rows.append(row)\n        if r[\'id\'] in duplicate_ids:signals.append(\'duplicacao_ou_semelhanca_em_train\')\n        if strong:signals.append(\'divergencia_semantica_forte_ia\')\n        if focus_difficult:signals.append(\'objeto_foco_sem_correspondencia_do_B\')\n        if sem:\n            sc=sem[\'scene\'];scene_counts[(r[\'source\'],sc[\'top\'],sc[\'indicative\'])]+=1\n            if sc[\'indicative\'] and sc[\'top\']!=\'interior\':signals.append(\'contexto_visual_possivelmente_fora_de_interior\')\n            for i,p in enumerate(sem[\'detector\'][\'extras\']):\n                obj=next((o for o in sem[\'objects\'] if o[\'kind\']==\'extra\' and o[\'bbox\']==p[\'bbox\'] and o[\'classe_id\']==p[\'classe_id\']),{})\n                possible_missing=bool(p[\'known_class\'] and p[\'best_iou_any_label\']<.1 and obj.get(\'status\')==\'compativel_ia\')\n                extra_rows.append({\'id\':r[\'id\'],\'source\':r[\'source\'],\'classe\':core.NAMES[p[\'classe_id\']],\n                    \'bbox_pred\':json.dumps(p[\'bbox\']),\'confidence\':p[\'confianca\'],\'cobertura_classe\':p[\'known_class\'],\n                    \'melhor_iou_qualquer_rotulo\':p[\'best_iou_any_label\'],\'triagem_recorte_ia\':obj.get(\'status\',\'nao_selecionado_para_clip\'),\n                    \'possivel_objeto_sem_caixa_na_classe_coberta\':possible_missing,\'interpretation\':p[\'interpretation\']})\n        status=(\'bloqueado_ou_quarentena\' if r[\'status_input\']!=\'legivel\' else\n                \'com_divergencia_semantica_forte\' if strong else \'com_incerteza_semantica\' if uncertain else \'sem_divergencia_forte_identificada\')\n        image_rows.append({\'id\':r[\'id\'],\'source\':r[\'source\'],\'lote\':r[\'lote\'],\'sha256\':r[\'sha256\'],\n            \'known_mask\':r[\'known_mask\'],\'status_entrada\':r[\'status_input\'],\'caixas_validas\':len(r[\'gt\']),\n            \'caixas_com_divergencia_forte_ia\':strong,\'caixas_inconclusivas_ou_pequenas\':uncertain,\n            \'objetos_foco_sem_correspondencia_do_B\':focus_difficult,\'triagem_imagem\':status,\n            \'sinais\':\'|\'.join(sorted(set(signals))),\'contexto_ia\':sem[\'scene\'][\'top\'] if sem else \'\',\n            \'contexto_ia_indicativo\':sem[\'scene\'][\'indicative\'] if sem else None,\n            \'revisao_humana_realizada\':False,\'rotulo_corrigido\':False})\n    core.save_csv(output/\'auditoria_imagens.csv\',image_rows)\n    core.save_csv(output/\'auditoria_caixas.csv\',box_rows)\n    core.save_csv(output/\'previsoes_sem_associacao.csv\',extra_rows)\n    core.save_csv(output/\'duplicacoes_treino.csv\',duplicates,[\'id\',\'similar_id\',\'type\',\'distance\'])\n    issues=[{\'id\':r[\'id\'],\'source\':r[\'source\'],**i} for r in records for i in r[\'issues\']]\n    fields=list(dict.fromkeys([\'id\',\'source\',\'line\',\'code\',\'blocking\']+[k for row in issues for k in row]))\n    core.save_csv(output/\'sinais_estruturais.csv\',issues,fields)\n    class_rows=[]\n    for source in sorted({r[\'source\'] for r in records}):\n        for cid,name in enumerate(core.NAMES):\n            subset=[b for b in box_rows if b[\'source\']==source and b[\'classe_id\']==cid]\n            counts=Counter(b[\'triagem_ia\'] for b in subset)\n            class_rows.append({\'source\':source,\'classe\':name,\'caixas_validas\':len(subset),\n                               \'compativeis_ia\':counts[\'compativel_ia\'],\'divergencias_fortes_ia\':counts[\'divergencia_forte_ia\'],\n                               \'inconclusivas_ia\':counts[\'inconclusivo_ia\'],\n                               \'pequenas_ou_invalidas_sem_analise\':counts[\'recorte_pequeno_sem_analise\']+counts[\'recorte_invalido_sem_analise\'],\n                               \'sem_analise_por_bloqueio\':counts[\'nao_analisado_por_bloqueio\'],\n                               \'sem_correspondencia_B\':sum(b[\'correspondencia_B_025\'] is False for b in subset)})\n    core.save_csv(output/\'resumo_por_classe.csv\',class_rows)\n    wardrobes=Counter((b[\'source\'],b[\'top_ia_recorte\'],b[\'triagem_ia\']) for b in box_rows if b[\'classe_id\']==7)\n    wardrobe_rows=[{\'source\':s,\'conceito_mais_proximo\':c or \'sem_analise\',\'triagem_ia\':st,\'caixas\':n}\n                   for (s,c,st),n in sorted(wardrobes.items())]\n    core.save_csv(output/\'wardrobe_hipoteses_semanticas.csv\',wardrobe_rows,[\'source\',\'conceito_mais_proximo\',\'triagem_ia\',\'caixas\'])\n    domains=[{\'source\':s,\'contexto_mais_proximo_ia\':c,\'margem_indicativa\':flag,\'imagens\':n} for (s,c,flag),n in sorted(scene_counts.items())]\n    core.save_csv(output/\'contextos_visuais_ia.csv\',domains)\n    source_counts=Counter(r[\'source\'] for r in records)\n    summary={\'versao\':core.VERSION,\'concluido\':True,\'status\':\'auditoria_concluida_sem_alteracao_de_dados\',\n             \'imagens_catalogadas\':len(records),\'imagens_por_fonte\':dict(source_counts),\n             \'imagens_analisadas_pelos_modelos\':len(semantics),\'status_entrada\':dict(Counter(r[\'status_input\'] for r in records)),\n             \'caixas_validas\':len(box_rows),\'triagem_caixas\':dict(Counter(r[\'triagem_ia\'] for r in box_rows)),\n             \'problemas_estruturais\':dict(Counter(r[\'code\'] for r in issues)),\n             \'pares_duplicacao_ou_semelhanca_registrados\':len(duplicates),\n             \'previsoes_sem_associacao_classe_coberta\':sum(r[\'cobertura_classe\'] for r in extra_rows),\n             \'previsoes_sem_associacao_classe_desconhecida\':sum(not r[\'cobertura_classe\'] for r in extra_rows),\n             \'possiveis_objetos_sem_caixa_triagem_ia\':sum(r[\'possivel_objeto_sem_caixa_na_classe_coberta\'] for r in extra_rows),\n             \'novos_treinamentos\':0,\'rotulos_alterados\':False,\'modelo_substituido\':False,\n             \'novo_dataset_aprovado\':False,\'preenchimento_manual_obrigatorio\':False,\'revisao_humana_realizada\':False,\n             \'test_home_extraido\':False,\'oi_test_extraido\':False,\'reserva_futura_extraida\':False,\n             \'validacoes_inferidas\':False,\'reserva_futura_inferida\':False,\n             \'proveniencia_treino_original_baseline\':\'nao_determinado\',\n             \'limites\':[\'Contagens são sinais automáticos; não estimam erros reais dos rótulos.\',\n                       \'Recorte Open Images dos cinco lotes do piloto; não representativo dos 80 mil.\',\n                       \'YOLO viu parte destes dados no treino; sua concordância não valida generalização.\',\n                       \'CLIP congelado pode errar; margens heurísticas não são probabilidades calibradas.\',\n                       \'pHash de 64 bits com distância até 4 registra até cinco vizinhos por imagem; é um sinal de semelhança e pode gerar falsos positivos.\',\n                       \'Home assume cobertura completa conforme protocolo anterior; a completude real não foi comprovada.\',\n                       \'pHash protegido externo usa apenas metadados já preservados; Home VAL/TEST e OI TEST têm proteção por SHA, sem nova revisão perceptual.\',\n                       \'Sem avaliação independente de risco físico ou utilidade para locomoção.\'],\n             \'execucao\':metadata}\n    core.save_json(output/\'resumo_auditoria_15.json\',summary)\n    # Galeria dirigida de até 48 imagens, estratificada por fonte e sinal.\n    lookup={r[\'id\']:r for r in records}; candidates=[r for r in image_rows if r[\'id\'] in semantics]\n    pools=defaultdict(list)\n    for row in candidates:\n        cat=(\'semantica\' if row[\'caixas_com_divergencia_forte_ia\'] else\n             \'dificuldade_detector\' if row[\'objetos_foco_sem_correspondencia_do_B\'] else\n             \'geometria_contexto\' if row[\'sinais\'] else \'comparacao_sem_sinal\')\n        pools[(row[\'source\'],cat)].append(row)\n    for key,rows in pools.items():rows.sort(key=lambda r:core.digest(r[\'id\']))\n    selected=[]\n    for i in range(48):\n        keys=sorted(pools)\n        if not keys:break\n        key=keys[i%len(keys)];selected.append((key,pools[key].pop(0)))\n        if not pools[key]:del pools[key]\n    cards=[];gallery=[]\n    for index,((source,category),row) in enumerate(selected):\n        relative=f\'galeria/caso_{index:03d}.jpg\';render_case(lookup[row[\'id\']],semantics[row[\'id\']],output/relative)\n        gallery.append({\'id\':row[\'id\'],\'source\':source,\'categoria\':category,\'arquivo\':relative})\n        cards.append(\'<article><h3>\'+html.escape(source+\' · \'+category)+\'</h3><p class="id">\'+html.escape(row[\'id\'])+\'</p><img loading="lazy" src="\'+relative+\'" alt="Caixas originais e recorte com contexto"><p>\'+html.escape(row[\'sinais\'] or \'Sem sinal automático nesta seleção.\')+\'</p></article>\')\n    core.save_json(output/\'indice_galeria.json\',gallery)\n    brief=f"Auditoria concluída: {len(records):,} imagens catalogadas; {len(semantics):,} analisadas pelos modelos; {len(box_rows):,} caixas válidas.".replace(\',\',\'.\')\n    focus_rows=[r for r in class_rows if r[\'classe\'] in {core.NAMES[c] for c in core.FOCUS}]\n    page=\'\'\'<!doctype html><html lang="pt-BR"><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>Auditoria 15 — Dados de treino</title><style>body{font:16px/1.55 system-ui;background:#eef3f8;color:#16263a;margin:0}main{max-width:1400px;margin:auto;padding:30px}h1{line-height:1.15}article,.note{background:white;padding:20px;border:1px solid #d9e3ed;border-radius:12px;margin:20px 0}img{width:100%;height:auto}table{border-collapse:collapse;background:white;width:100%}th,td{border:1px solid #d9e3ed;padding:10px;text-align:left}.scroll{overflow:auto}.id{font-size:12px;overflow-wrap:anywhere}a{color:#1163aa}</style><main><p>TCC · Visão computacional</p><h1>Auditoria automática dos dados de treino</h1>\'\'\'\n    page+=\'<div class="note"><strong>\'+brief+\'</strong><p>Nenhum rótulo, modelo ou conjunto de avaliação foi alterado. Os sinais abaixo orientam o próximo experimento; não são uma taxa de erro confirmada das anotações.</p></div>\'\n    page+=\'<h2>Classes prioritárias</h2>\'+html_table(focus_rows)\n    page+=\'<p>Todas as 12 classes foram auditadas nas imagens elegíveis. A divergência forte exige o mesmo conceito alternativo nos dois recortes e margem mínima congelada; concordância com CLIP também não comprova o rótulo.</p>\'\n    page+=\'<h2>Hipóteses para wardrobe</h2><p>A hipótese estreita é um móvel alto para roupas. Armários genéricos, estantes, racks e cômodas são conceitos alternativos de triagem, sem alteração da taxonomia original.</p>\'+html_table(wardrobe_rows)\n    page+=\'<h2>Contexto visual — hipótese por IA</h2>\'+html_table(domains)\n    page+=\'<h2>Limites</h2><ul>\'+\'\'.join(\'<li>\'+html.escape(x)+\'</li>\' for x in summary[\'limites\'])+\'</ul>\'\n    page+=\'<h2>Próximo passo</h2><p>Enviar o pacote desta auditoria para decidir um recorte ou uma correção documentada antes de outro treino. Não há formulário para preencher. Uma seleção futura pode preservar rótulos e máscaras e remover imagens inteiras; sinais CLIP não serão tratados como novas anotações verdadeiras.</p>\'\n    page+=\'<h2>Exemplos selecionados automaticamente</h2><p>A galeria é dirigida por sinais e não estima sua frequência na base.</p>\'+\'\'.join(cards)+\'</main></html>\'\n    (output/\'relatorio_auditoria_15.html\').write_text(page,encoding=\'utf-8\')\n    (output/\'LEIA_RESULTADO.md\').write_text(\'# Resultado do notebook 15\\n\\n\'+brief+\'\\n\\nResultados em pacote_auditoria_15.zip. Não é necessário preencher fichas ou repetir os testes.\\n\\nAbra relatorio_auditoria_15.html após extrair o ZIP. As contagens descrevem sinais por IA e verificações estruturais, não erros de rótulo confirmados. Os originais permanecem preservados; não há novo dataset aprovado ou treinamento.\\n\',encoding=\'utf-8\')\n    return summary\n'}
CODIGO_SHA256 = {'audit15_core.py': '80c4a2e88f6d7c19133968bcf07dc0928a14fa174606fd13f2e0a5c813efa53e', 'audit15_models.py': 'a5ac97cb14b7703c1ad73cfeab198004f07aa1fd697c3eea733b4c6e247d33d7', 'audit15_reports.py': 'acc1ccc46d3686a57882d55ffe2d6d75fab7f8e639a34c476de3e36458287641'}
PASTA_CODIGO=PASTA_LOCAL/'codigo';PASTA_CODIGO.mkdir(exist_ok=True)
for nome,fonte in MODULOS_EMBUTIDOS.items():
    if hashlib.sha256(fonte.encode()).hexdigest()!=CODIGO_SHA256[nome]:raise RuntimeError('Código embutido alterado.')
    compile(fonte,nome,'exec');(PASTA_CODIGO/nome).write_text(fonte,encoding='utf-8')
    (PASTA_RESULTADOS/'codigo').mkdir(exist_ok=True)
    (PASTA_RESULTADOS/'codigo'/nome).write_text(fonte,encoding='utf-8')
sys.path.insert(0,str(PASTA_CODIGO))
import audit15_core as core
import audit15_models as modelos
import audit15_reports as relatorios

def registrar_falha(exc,etapa):
    core.save_json(PASTA_RESULTADOS/'execucao_interrompida.json',{
        'concluido':False,'etapa':etapa,'tipo':type(exc).__name__,'mensagem':str(exc)[:1200],
        'rotulos_alterados':False,'novos_treinamentos':0,'preenchimento_manual_obrigatorio':False})
    pacote=core.package(PASTA_RESULTADOS)
    if 'enviar_resultados_drive' in globals():
        try:enviar_resultados_drive('diagnóstico parcial',obrigatorio=False)
        except Exception:pass
    print('Execução interrompida. O pacote parcial registra o motivo:',pacote,flush=True)
    display(FileLink(str(pacote)))

try:
    import clip
    raiz_clip=Path(clip.__file__).parent
    for nome,sha in CLIP_ARQUIVOS_SHA256.items():
        if core.sha_file(raiz_clip/nome)!=sha:raise RuntimeError('A instalação CLIP difere do código congelado: '+nome)
    AMBIENTE={'python':platform.python_version(),**modelos.environment(),'device':str(DEVICE)}
    print('Conferindo e carregando o CLIP. O primeiro download tem cerca de 354 MB.',flush=True)
    CLIP_SCORER=modelos.ClipScorer(DEVICE,Path('/tmp/tcc_clip_15'))
    from PIL import Image
    teste_scores=CLIP_SCORER.score([Image.new('RGB',(224,224),(127,127,127))])
    if teste_scores.shape!=(1,len(modelos.CONCEPTS)):raise RuntimeError('Saída CLIP incompatível.')
    core.save_json(PASTA_RESULTADOS/'ambiente.json',AMBIENTE)
    print('Código e peso CLIP conferidos. A análise semântica está pronta.')
except Exception as exc:
    registrar_falha(exc,'ambiente e CLIP');raise


## 3. Conectar ao Drive e autenticar os manifestos

Reutiliza o Secret já configurado. Os hashes do pacote Home, preparação,
divisão histórica, lotes, pacote 11 e peso B precisam conferir antes da análise.
O pacote 11 fornece apenas o manifesto de identidades da validação/reserva;
as imagens externas de validação/reserva não serão baixadas ou extraídas.


In [ ]:
from pathlib import Path
import base64
import configparser
import json
import shutil
import subprocess
import urllib.request
import zipfile
from kaggle_secrets import UserSecretsClient

SECRET_DRIVE = "GDRIVE_RCLONE_CONFIG_B64"
REMOTE_DRIVE = "gdrive_tcc"
ORIGEM_DRIVE = f"{REMOTE_DRIVE}:TCC_Visao_Computacional"
DRIVE_TCC = Path("/tmp/tcc_100k_entradas")
ENVIAR_RESULTADOS_AO_DRIVE = True
EXECUTAR_CALIBRACAO = False

# Credenciais ficam fora da pasta de resultados salva pelo Kaggle.
PASTA_RCLONE = Path("/tmp/tcc_rclone")
PASTA_RCLONE.mkdir(mode=0o700, parents=True, exist_ok=True)
PASTA_RCLONE.chmod(0o700)
CONFIG_RCLONE = PASTA_RCLONE / "gdrive_tcc.conf"
RCLONE = PASTA_RCLONE / "rclone"

try:
    segredo = UserSecretsClient().get_secret(SECRET_DRIVE)
except Exception:
    raise RuntimeError(
        f"Crie o Secret {SECRET_DRIVE} em Add-ons → Secrets e habilite "
        "o acesso deste notebook. Siga a preparação descrita acima."
    ) from None
try:
    configuracao_bytes = base64.b64decode(segredo.strip(), validate=True)
    configuracao = configparser.ConfigParser(interpolation=None)
    configuracao.read_string(configuracao_bytes.decode("utf-8-sig"))
except Exception:
    raise RuntimeError(
        "O Secret não contém a configuração rclone codificada corretamente. "
        "Copie novamente pelo comando PowerShell e atualize o Secret."
    ) from None
if (not configuracao.has_section(REMOTE_DRIVE)
        or configuracao.get(REMOTE_DRIVE, "type", fallback="") != "drive"
        or not configuracao.get(REMOTE_DRIVE, "token", fallback="")
        or not configuracao.get(REMOTE_DRIVE, "client_id", fallback="")):
    raise RuntimeError(
        "A configuração precisa da conexão gdrive_tcc, tipo drive, "
        "cliente OAuth próprio e autorização concluída no navegador."
    )
if (ENVIAR_RESULTADOS_AO_DRIVE
        and configuracao.get(REMOTE_DRIVE, "scope", fallback="") != "drive"):
    raise RuntimeError(
        "Para salvar os resultados, configure scope 1 (drive, leitura e "
        "gravação) no rclone e atualize o Secret."
    )
# Esta conexão usa Meu Drive, não um Drive compartilhado.
configuracao.remove_option(REMOTE_DRIVE, "team_drive")
with CONFIG_RCLONE.open("w", encoding="utf-8") as arquivo_config:
    configuracao.write(arquivo_config)
CONFIG_RCLONE.chmod(0o600)
del segredo, configuracao_bytes, configuracao

if not RCLONE.is_file():
    import os

    executavel = shutil.which("rclone")
    if executavel is None:
        prefixo = [] if os.geteuid() == 0 else ["sudo"]
        subprocess.run(prefixo + ["apt-get", "update", "-qq"], check=True)
        subprocess.run(
            prefixo + ["apt-get", "install", "-y", "--no-install-recommends", "rclone"],
            check=True,
        )
        executavel = shutil.which("rclone")

    if executavel is None:
        raise RuntimeError("A instalação terminou, mas o executável rclone não foi localizado.")

    shutil.copy2(executavel, RCLONE)
    RCLONE.chmod(0o700)
    print("rclone instalado pelo gerenciador de pacotes do Linux.")

DRIVE_TCC.mkdir(parents=True, exist_ok=True)
if shutil.disk_usage(DRIVE_TCC).free < 10 * 1024**3:
    raise RuntimeError("Reserve pelo menos 10 GiB livres para baixar as entradas.")


def baixar_arquivo_drive(relativo,destino):
    destino=Path(destino);destino.parent.mkdir(parents=True,exist_ok=True)
    recibo=destino.with_name(destino.name+'.download.json')
    if destino.is_file() and recibo.is_file():
        anterior=json.loads(recibo.read_text())
        if anterior.get('origem')==relativo and anterior.get('sha256')==core.sha_file(destino):
            print('Entrada local conferida:',relativo,flush=True);return destino
    print('Baixando:',relativo,flush=True)
    temporario=destino.with_name(destino.name+'.part')
    subprocess.run([str(RCLONE),'--config',str(CONFIG_RCLONE),'copyto',
        ORIGEM_DRIVE+'/'+relativo,str(temporario),'--retries','3','--low-level-retries','5',
        '--contimeout','15s','--timeout','60s'],check=True)
    if not temporario.is_file() or not temporario.stat().st_size:raise RuntimeError('Download vazio: '+relativo)
    temporario.replace(destino)
    core.save_json(recibo,{'origem':relativo,'sha256':core.sha_file(destino),'bytes':destino.stat().st_size})
    return destino

def enviar_resultados_drive(etapa="manual", obrigatorio=False):
    if not ENVIAR_RESULTADOS_AO_DRIVE:
        return False
    pasta = Path(PASTA_RESULTADOS)
    destino = f"{ORIGEM_DRIVE}/resultados/{NOME_AUDITORIA15}"
    print(f"Enviando resultados ao Drive ({etapa})...", flush=True)
    try:
        enviado = subprocess.run(
            [str(RCLONE), "--config", str(CONFIG_RCLONE), "copy",
             str(pasta), destino, "--exclude", "**/*.tmp",
             "--retries", "2", "--low-level-retries", "3",
             "--contimeout", "15s", "--timeout", "60s"],
            check=False,
        )
        if enviado.returncode != 0:
            raise RuntimeError(f"rclone retornou código {enviado.returncode}")
    except (OSError, RuntimeError) as exc:
        mensagem = (
            f"Envio ao Drive falhou em {etapa}: {exc}. "
            "Os arquivos locais continuam em /kaggle/working/resultados_tcc. "
            "Confira a autorização e execute a última etapa para repetir o envio."
        )
        if obrigatorio:
            raise RuntimeError(mensagem) from None
        print(mensagem, flush=True)
        return False
    print("Resultados copiados ao Drive.", flush=True)
    return True

def enviar_resultados(etapa,obrigatorio=True):
    return enviar_resultados_drive(etapa,obrigatorio=obrigatorio)


ENTRADAS=PASTA_LOCAL/'entradas'
OI_RELATIVO='datasets/openimages_v7_tcc_100k_v1'
PASTA_EXECUCAO_10='resultados/yolov8n_home_openimages_80_20_1ep_kaggle_20261015_030228_068010'
PASTA_EXECUCAO_11='resultados/diagnostico11_20261016_024603_683411'
PASTA_EXECUCAO_12='resultados/ajuste12_20261016_033819_596490'
try:
    PREP=baixar_arquivo_drive(OI_RELATIVO+'/configuracao_preparacao.json',ENTRADAS/'configuracao_preparacao.json')
    PACOTES=baixar_arquivo_drive(OI_RELATIVO+'/manifesto_pacotes.json',ENTRADAS/'manifesto_pacotes.json')
    DIVISAO_CSV=baixar_arquivo_drive(PASTA_EXECUCAO_10+'/divisao_openimages.csv',ENTRADAS/'divisao_10.csv')
    DIVISAO_JSON=baixar_arquivo_drive(PASTA_EXECUCAO_10+'/divisao_openimages.json',ENTRADAS/'divisao_10.json')
    PACOTE_11=baixar_arquivo_drive(PASTA_EXECUCAO_11+'/pacote_diagnostico_11.zip',ENTRADAS/'pacote_11.zip')
    cfg=json.loads(PREP.read_text())
    if core.digest(cfg)!=EXPECTED['preparacao'] or cfg['nomes']!=core.NAMES:raise RuntimeError('Preparação alterada.')
    if core.sha_file(PACOTES)!=EXPECTED['manifesto_pacotes']:raise RuntimeError('Manifesto dos lotes alterado.')
    HISTORICO=core.load_history(DIVISAO_CSV,DIVISAO_JSON,EXPECTED['divisao_10'])
    HISTORICO_MAP={r['id']:r for r in HISTORICO}
    with zipfile.ZipFile(PACOTE_11) as z:
        core.inspect_zip(z)
        if core.sha_file(PACOTE_11)!=EXPECTED['pacote_11']:raise RuntimeError('Pacote 11 alterado.')
        MANIFESTO_NOVO=json.loads(z.read('manifesto_validacao_nova.json'))
        c11=json.loads(z.read('configuracao_diagnostico.json'))
        s11=json.loads(z.read('resumo_diagnostico_11.json'))
    base_manifesto=dict(MANIFESTO_NOVO);declarado=base_manifesto.pop('dataset_sha256')
    if (declarado!=EXPECTED['dataset_novo'] or core.digest(base_manifesto)!=declarado
        or MANIFESTO_NOVO['nomes']!=core.NAMES or MANIFESTO_NOVO['origem_split']!='validation'
        or c11['dataset_sha256']!=declarado or c11['politica_sha256']!=EXPECTED['politica']
        or s11['reserva_futura_inferida'] is not False):raise RuntimeError('Manifesto externo incompatível.')
    itens_novos=MANIFESTO_NOVO['arquivos']
    if (Counter(r['role'] for r in itens_novos)!={'val_nova':1000,'reserva_futura':300}
        or len({r['id'] for r in itens_novos})!=1300
        or len({r['sha256'] for r in itens_novos})!=1300):raise RuntimeError('Identidades externas incompatíveis.')
    pacotes=json.loads(PACOTES.read_text())
    if pacotes['config_sha256']!=EXPECTED['preparacao']:raise RuntimeError('Configuração dos lotes incompatível.')
    nomes_lotes={f'lote_{i:06}.zip' for i in range(1,6)}
    LOTES=sorted([r for r in pacotes['lotes'] if r['arquivo'] in nomes_lotes],key=lambda r:r['arquivo'])
    RECORTE_HISTORICO=[r for r in HISTORICO if r['lote'] in nomes_lotes]
    if (len(LOTES)!=5 or len(RECORTE_HISTORICO)!=5000 or any(r['split']!='train' for r in RECORTE_HISTORICO)):
        raise RuntimeError('Os cinco lotes precisam conter exatamente 5.000 IDs históricos TRAIN.')
    core.save_csv(PASTA_RESULTADOS/'distribuicao_historico_e_recorte.csv',
                  core.history_distribution(HISTORICO,{r['id'] for r in RECORTE_HISTORICO}))
    ZIP_HOME=baixar_arquivo_drive('datasets/homeobjects_sunrgbd_validado.zip',ENTRADAS/'home.zip')
    PESO_B=baixar_arquivo_drive(PASTA_EXECUCAO_12+'/pesos_checkpoints/B_home_replay_e03.pt',ENTRADAS/'B_home_replay_e03.pt')
    if core.sha_file(PESO_B)!=EXPECTED['B_e03']:raise RuntimeError('Checkpoint B diferente do avaliado.')
    CONGELADOS={str(p):core.sha_file(p) for p in [PREP,PACOTES,DIVISAO_CSV,DIVISAO_JSON,PACOTE_11,ZIP_HOME,PESO_B]}
    print('Entradas autenticadas. As imagens dos conjuntos de avaliação permanecem fora da análise.')
except Exception as exc:
    registrar_falha(exc,'entradas e manifestos');raise


## 4. Auditar o treino e registrar duplicações

Cada linha de anotação é verificada, inclusive classes fora do foco. O código
registra caixas inválidas, pequenas, sobrepostas e conflitos de cobertura.
Erros de identidade/hash interrompem a execução; problemas de rótulo entram no relatório.

Coincidências exatas com conjuntos protegidos são bloqueadas antes da extração
da imagem de treino. Semelhanças pHash com as identidades externas já preservadas
ficam em quarentena antes da inferência. Para Home VAL/TEST e Open Images TEST,
a proteção disponível aqui é por SHA, sem abrir as imagens para um novo pHash.

Os cinco lotes são baixados um por vez e removidos de /tmp após a extração.
Nenhum original no Drive é alterado. As duplicações dentro do treino são apenas
sinalizadas nesta etapa, sem apagar exemplos.


In [ ]:
try:
    HOME_CATALOGO,HOME_PROTEGIDOS=core.home_catalog(ZIP_HOME,EXPECTED)
    PROTEGIDOS=core.protected_index(HISTORICO,HOME_PROTEGIDOS,MANIFESTO_NOVO)
    REGISTROS=core.extract_home(ZIP_HOME,HOME_CATALOGO,PASTA_LOCAL/'dados',PROTEGIDOS,progress=print)
    RECIBOS_LOTES=[]
    for lote in LOTES:
        if shutil.disk_usage(PASTA_LOCAL).free < lote['zip_bytes']+512*1024**2:
            raise RuntimeError('Espaço insuficiente para o próximo lote. Os resultados parciais estão preservados.')
        caminho=baixar_arquivo_drive(OI_RELATIVO+'/lotes/'+lote['arquivo'],ENTRADAS/lote['arquivo'])
        grupo=core.extract_oi_package(caminho,lote,HISTORICO_MAP,EXPECTED['preparacao'],PASTA_LOCAL/'dados',PROTEGIDOS)
        REGISTROS.extend(grupo);RECIBOS_LOTES.append(lote)
        caminho.unlink();caminho.with_name(caminho.name+'.download.json').unlink(missing_ok=True)
        core.save_gz(PASTA_RESULTADOS/'manifesto_imagens_auditadas.json.gz',core.manifest_public(REGISTROS))
        core.save_json(PASTA_RESULTADOS/'progresso_dados.json',{'imagens_catalogadas':len(REGISTROS),'lotes_conferidos':len(RECIBOS_LOTES)})
        enviar_resultados('auditoria estrutural: '+lote['arquivo'])
        print('Lote auditado:',lote['arquivo'],'| catalogadas até agora:',len(REGISTROS),flush=True)
    if len(REGISTROS)!=6824 or len({r['id'] for r in REGISTROS})!=6824:raise RuntimeError('Catálogo incompleto ou repetido.')
    DUPLICACOES=core.duplicate_rows(REGISTROS)
    core.save_csv(PASTA_RESULTADOS/'duplicacoes_treino.csv',DUPLICACOES,['id','similar_id','type','distance'])
    ELEGIVEIS=core.eligible(REGISTROS)
    if not ELEGIVEIS:raise RuntimeError('Nenhuma imagem elegível para a análise por modelos; confira os bloqueios no pacote parcial.')
    core.save_json(PASTA_RESULTADOS/'recibos_lotes.json',RECIBOS_LOTES)
    print('Catálogo:',len(REGISTROS),'| elegíveis para modelos:',len(ELEGIVEIS),
          '| bloqueadas/quarentena:',len(REGISTROS)-len(ELEGIVEIS))
except Exception as exc:
    if 'REGISTROS' in globals():core.save_gz(PASTA_RESULTADOS/'manifesto_imagens_auditadas.json.gz',core.manifest_public(REGISTROS))
    registrar_falha(exc,'auditoria estrutural do treino');raise


## 5. Congelar as regras e recuperar caches compatíveis

Os sinais semânticos não alteram a confiança do modelo em uso. O B captura
propostas em 0,15 para diagnóstico; a associação de treino usa 0,25 e IoU 0,50.
Não são calculadas métricas de generalização a partir da concordância em treino.

O CLIP examina todas as caixas válidas de todas as classes nas imagens elegíveis.
Também examina até dez previsões sem associação por imagem nas quatro classes
prioritárias, com confiança pelo menos 0,25. Cobertura desconhecida continua desconhecida.


In [ ]:
try:
    PROTOCOLO={'versao':core.VERSION,'seed':SEED,'classes':core.NAMES,
        'foco':sorted(core.FOCUS),'home_catalogo':1824,'openimages_catalogo':5000,
        'lotes':sorted(nomes_lotes),'peso_B':EXPECTED['B_e03'],'clip_sha256':modelos.CLIP_SHA,
        'clip_commit':modelos.CLIP_COMMIT,'predict_args':modelos.PREDICT_ARGS,
        'iou_associacao_treino':.5,'confianca_associacao_treino':.25,
        'semantica':modelos.SEMANTIC_RULE,'conceitos':modelos.CONCEPTS,'templates':modelos.TEMPLATES,
        'deduplicacao':{'phash_bits':64,'distancia_maxima':4,'vizinhos_maximos_registrados_por_imagem':5,
                       'home_val_test_e_oi_test':'somente_SHA_sem_novo_phash'},
        'cenas':modelos.SCENES,'revisao_humana_realizada':False,
        'alteracao_rotulos':False,'treinamento':False,'preenchimento_manual_obrigatorio':False,
        'teste_ou_reserva_inferidos':False,'home_cobertura_completa':'hipotese_herdada_nao_comprovada',
        'scope_limit':'cinco lotes do piloto, sem representatividade garantida dos 80 mil',
        'resultado_14':'nenhum dos sete limiares globais elegivel; baseline preservado',
        'baseline_proveniencia_original':'nao_determinado'}
    DATA_SHA=core.digest(core.manifest_public(REGISTROS))
    CONTEXTO={'ambiente':AMBIENTE,'codigo_sha256':CODIGO_SHA256,'dados_sha256':DATA_SHA,'protocolo_sha256':core.digest(PROTOCOLO)}
    CACHE_KEY=core.digest(CONTEXTO)
    CACHE_DIR=PASTA_RESULTADOS/'caches';CACHE_DIR.mkdir(exist_ok=True)
    CACHE_REMOTE=ORIGEM_DRIVE+'/resultados/auditoria15_cache_'+CACHE_KEY[:24]
    core.save_json(PASTA_RESULTADOS/'protocolo_congelado_15.json',PROTOCOLO)
    core.save_json(PASTA_RESULTADOS/'contexto_execucao.json',{'cache_key':CACHE_KEY,**CONTEXTO})
    checagem=subprocess.run([str(RCLONE),'--config',str(CONFIG_RCLONE),'lsjson',CACHE_REMOTE,'--stat'],capture_output=True,text=True)
    if checagem.returncode==0:
        print('Recuperando caches do mesmo protocolo, dados, pesos, código e ambiente...',flush=True)
        subprocess.run([str(RCLONE),'--config',str(CONFIG_RCLONE),'copy',CACHE_REMOTE,str(CACHE_DIR),
            '--exclude','rejeitados/**','--exclude','**/rejeitados/**','--retries','3','--low-level-retries','5',
            '--contimeout','15s','--timeout','60s'],check=True)
    elif checagem.returncode in {3,4}:
        print('Primeira passagem neste contexto: os caches serão construídos.')
    else:raise RuntimeError('Não foi possível verificar os caches no Drive; código rclone '+str(checagem.returncode))

    def salvar_checkpoint_cache(etapa):
        core.save_json(PASTA_RESULTADOS/'progresso_modelos.json',{
            'etapa':etapa,'atualizado_em':datetime.now(timezone.utc).isoformat(),'cache_key':CACHE_KEY})
        # A cópia transfere apenas arquivos novos ou modificados. Originais nunca são tocados.
        subprocess.run([str(RCLONE),'--config',str(CONFIG_RCLONE),'copy',str(CACHE_DIR),CACHE_REMOTE,
            '--exclude','**/*.tmp','--exclude','**/rejeitados/**',
            '--retries','2','--low-level-retries','3','--contimeout','15s','--timeout','60s'],check=True)

    # O significado original do MID fica documentado separadamente da hipótese estreita de wardrobe.
    taxonomia={'nomes_originais':cfg['nomes'],'mids_raizes_originais':cfg['mids_raizes'],
        'mapeamento_alterado':False,'hipotese_wardrobe':'móvel alto destinado a roupas',
        'alternativas_somente_triagem':[c for c,_ in modelos.CONCEPTS[12:]],
        'fonte_classe_oficial':'https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions-boxable.csv'}
    try:
        import urllib.request,csv,io
        with urllib.request.urlopen(taxonomia['fonte_classe_oficial'],timeout=45) as resposta: raw=resposta.read()
        tabela=dict(csv.reader(io.StringIO(raw.decode('utf-8-sig'))))
        taxonomia['rotulos_oficiais_raizes']=[{'classe':n,'mid':m,'descricao_oficial':tabela.get(m)} for n,m in zip(cfg['nomes'],cfg['mids_raizes'])]
        taxonomia['descricoes_oficiais_sha256']=hashlib.sha256(raw).hexdigest()
    except Exception as exc:
        taxonomia['descricoes_oficiais_status']='nao_recuperadas; MIDs originais preservados'
        taxonomia['falha_recuperacao']=type(exc).__name__
    core.save_json(PASTA_RESULTADOS/'taxonomia_documentada.json',taxonomia)
    CONFIG={'versao':core.VERSION,'experimento':NOME_AUDITORIA15,'ambiente':AMBIENTE,
            'protocolo_sha256':core.digest(PROTOCOLO),'dados_sha256':DATA_SHA,'codigo_sha256':CODIGO_SHA256,
            'entradas_sha256':CONGELADOS,'lotes_autenticados':RECIBOS_LOTES,'cache_key':CACHE_KEY,
            'reserva_futura_extraida':False,'validacoes_inferidas':False,'testes_inferidos':False,
            'novos_treinamentos':0,'rotulos_alterados':False}
    core.save_json(PASTA_RESULTADOS/'configuracao_auditoria_15.json',CONFIG)
    enviar_resultados('protocolo congelado antes das novas inferências de treino')
except Exception as exc:
    registrar_falha(exc,'protocolo e retomada');raise


## 6. Verificar concordância com o B nas imagens de treino

Uma passagem por imagem elegível, com cache por bloco de até 100. Caixas
numéricas de área zero são preservadas no cache e contabilizadas, sem repetir
o erro da etapa 6 do notebook anterior. Elas não entram na associação ou nos recortes.

Uma caixa anotada sem correspondência pode ser difícil para o detector; isso
não a torna incorreta. Uma previsão sem associação em classe coberta também é
um caso a investigar, não uma confirmação de anotação ausente.


In [ ]:
try:
    PREDICOES,RESUMO_YOLO=modelos.infer_yolo(ELEGIVEIS,PESO_B,EXPECTED['B_e03'],CACHE_DIR,
        CONTEXTO,DEVICE,checkpoint=salvar_checkpoint_cache)
    core.save_json(PASTA_RESULTADOS/'resumo_inferencia_yolo.json',RESUMO_YOLO)
    enviar_resultados('concordância do detector em treino concluída')
except Exception as exc:
    registrar_falha(exc,'inferência YOLO somente TRAIN');raise


## 7. Analisar automaticamente os recortes e o contexto visual

CLIP fornece os conceitos mais próximos em duas vistas de cada caixa.
As saídas são `compativel_ia`, `divergencia_forte_ia`, `inconclusivo_ia` ou
recorte insuficiente. Nenhuma dessas categorias equivale a revisão humana.
As hipóteses de contexto — interior, exterior, produto, colagem ou detalhe —
são descritivas e não excluem imagens automaticamente.

O código imprime progresso dentro dos blocos e faz backup ao fechar cada bloco.


In [ ]:
try:
    SEMANTICA,RESUMO_CLIP=modelos.infer_semantics(ELEGIVEIS,PREDICOES,CACHE_DIR,CONTEXTO,
        DEVICE,Path('/tmp/tcc_clip_15'),checkpoint=salvar_checkpoint_cache,scorer=CLIP_SCORER)
    core.save_json(PASTA_RESULTADOS/'resumo_inferencia_clip.json',RESUMO_CLIP)
    enviar_resultados('triagem automática de todas as classes concluída')
except Exception as exc:
    registrar_falha(exc,'análise semântica somente TRAIN');raise


## 8. Consolidar os diagnósticos e gerar o pacote

O pacote inclui contagens por classe e fonte, diagnósticos de geometria e
cobertura, duplicações, hipóteses semânticas de `wardrobe`, contextos visuais,
previsões sem associação, caches, protocolo e uma galeria automática.

Os relatórios diferenciam problemas técnicos, incerteza por IA e dificuldade
do detector. Não será criado um dataset rotulado como aprovado a partir da IA.
Depois desta execução, basta enviar o ZIP para definirmos o experimento seguinte.


In [ ]:
try:
    core.assert_inputs_unchanged(REGISTROS)
    for caminho,sha in CONGELADOS.items():
        if core.sha_file(caminho)!=sha:raise RuntimeError('Entrada protegida alterada: '+caminho)
    if set(PREDICOES)!={r['id'] for r in ELEGIVEIS} or set(SEMANTICA)!=set(PREDICOES):
        raise RuntimeError('Modelos não concluíram todas as imagens elegíveis.')
    CONFIG['inferencia_yolo']=RESUMO_YOLO;CONFIG['inferencia_clip']=RESUMO_CLIP
    core.save_json(PASTA_RESULTADOS/'configuracao_auditoria_15.json',CONFIG)
    RESUMO=relatorios.report(PASTA_RESULTADOS,REGISTROS,SEMANTICA,PREDICOES,DUPLICACOES,CONFIG)
    core.save_json(PASTA_RESULTADOS/'integridade_final.json',{
        'conferida':True,'catalogadas':len(REGISTROS),'elegiveis_modelos':len(ELEGIVEIS),
        'rotulos_alterados':False,'novos_treinamentos':0,'modelo_substituido':False,
        'test_home_extraido':False,'oi_test_extraido':False,'reserva_futura_extraida':False,
        'validacoes_inferidas':False,'reserva_futura_inferida':False,
        'proveniencia_original_baseline':'nao_determinado'})
    erro=PASTA_RESULTADOS/'execucao_interrompida.json'
    if erro.is_file():
        historico=PASTA_RESULTADOS/'historico_falhas';historico.mkdir(exist_ok=True)
        erro.replace(historico/('falha_resolvida_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')+'.json'))
    PACOTE_FINAL=core.package(PASTA_RESULTADOS)
    enviar_resultados('pacote final da auditoria 15')
    print('NOTEBOOK 15 CONCLUÍDO — sem treino, sem alteração de rótulos e sem formulários.')
    print('Imagens catalogadas:',RESUMO['imagens_catalogadas'],'| analisadas por modelos:',RESUMO['imagens_analisadas_pelos_modelos'])
    print('Arquivo de resultados:',PACOTE_FINAL)
    display(FileLink(str(PACOTE_FINAL)))
    display(FileLink(str(PASTA_RESULTADOS/'relatorio_auditoria_15.html')))
except Exception as exc:
    registrar_falha(exc,'relatórios e salvamento final');raise
